In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Load dataset
df = pd.read_csv("../../data/student_queries.csv")

# Encode labels
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["intent"])

df.head()

,datetime,student,question,intent,is_distressed,label
0,2025-07-04 14:07:23,Shannon Austin,What topics will be covered in the AI course?,Course Information,0,4
1,2025-07-04 16:13:28,Stephanie Calhoun,I'm feeling really overwhelmed lately.,Mental Health Concerns,1,9
2,2025-07-04 16:19:28,Kevin Garcia,Who do I contact for transcript requests?,Administrative Support,0,1
3,2025-07-04 16:55:20,Lisa Duran,I have a complaint about the cafeteria service.,General Complaint,1,7
4,2025-07-04 17:08:35,Jeff Rangel,Who do I contact for transcript requests?,Administrative Support,0,1


In [2]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Train-test split
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)

# Convert to Hugging Face Dataset format
train_dataset = Dataset.from_pandas(train_df[["question", "label"]])
test_dataset = Dataset.from_pandas(test_df[["question", "label"]])

/Users/bennyxiong/Documents/Source/ML/CSCN8010_FinalProject/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/bennyxiong/Documents/Source/ML/CSCN8010_FinalProject/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import torch

# Tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["question"], padding=True, truncation=True)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

# Model setup
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=len(label_encoder.classes_)
)

# Metrics
def compute_metrics(pred):
    logits, labels = pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
        "precision": precision_score(labels, preds, average="weighted"),
        "recall": recall_score(labels, preds, average="weighted"),
    }

# Training arguments
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

/Users/bennyxiong/Documents/Source/ML/CSCN8010_FinalProject/.venv/lib/python3.9/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 300/300 [00:00<00:00, 9346.99 examples/s]
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.bias', 'cls.predictions.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSe

In [4]:
# Train the model
trainer.train()

# Evaluate on test set
trainer.evaluate()

The following columns in the training set don't have a corresponding argument in `BertForSequenceClassification.forward` and have been ignored: __index_level_0__, question. If __index_level_0__, question are not expected by `BertForSequenceClassification.forward`,  you can safely ignore this message.
/Users/bennyxiong/Documents/Source/ML/CSCN8010_FinalProject/.venv/lib/python3.9/site-packages/transformers/optimization.py:306: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
***** Running training *****
  Num examples = 1200
  Num Epochs = 10
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 1
  Total optimization steps = 750
  Number of trainable parameters = 109493775
  0%|          | 0/750 [00:00<?, ?it/s]

{'eval_loss': 1.116915225982666, 'eval_accuracy': 1.0, 'eval_f1': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_runtime': 1.1224, 'eval_samples_per_second': 267.277, 'eval_steps_per_second': 4.455, 'epoch': 1.0}


Model weights saved in ./results/checkpoint-75/pytorch_model.bin
/Users/bennyxiong/Documents/Source/ML/CSCN8010_FinalProject/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
 13%|█▎        | 100/750 [00:34<03:33,  3.05it/s]

{'loss': 1.7478, 'learning_rate': 1.7333333333333336e-05, 'epoch': 1.33}


 20%|██        | 150/750 [00:51<03:13,  3.11it/s]The following columns in the evaluation set don't have a corresponding argument in `BertForSequenceClassification.forward` and have been ignored: __index_level_0__, question. If __index_level_0__, question are not expected by `BertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 300
  Batch size = 64
                                                 
 20%|██        | 150/750 [00:52<03:13,  3.11it/s]Saving model checkpoint to ./results/checkpoint-150
Configuration saved in ./results/checkpoint-150/config.json


...

{'eval_loss': 0.04922822490334511, 'eval_accuracy': 1.0, 'eval_f1': 1.0, 'eval_precision': 1.0, 'eval_recall': 1.0, 'eval_runtime': 1.1514, 'eval_samples_per_second': 260.547, 'eval_steps_per_second': 4.342, 'epoch': 3.0}


Model weights saved in ./results/checkpoint-225/pytorch_model.bin


Training completed. Do not forget to share your model on huggingface.co/models =)


Loading best model from ./results/checkpoint-75 (score: 1.0).
 30%|███       | 225/750 [01:20<03:07,  2.81it/s]
The following columns in the evaluation set don't have a corresponding argument in `BertForSequenceClassification.forward` and have been ignored: __index_level_0__, question. If __index_level_0__, question are not expected by `BertForSequenceClassification.forward`,  you can safely ignore this message.
***** Running Evaluation *****
  Num examples = 300
  Batch size = 64
/Users/bennyxiong/Documents/Source/ML/CSCN8010_FinalProject/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


{'train_runtime': 80.2152, 'train_samples_per_second': 149.598, 'train_steps_per_second': 9.35, 'train_loss': 0.9375639777713352, 'epoch': 3.0}


100%|██████████| 5/5 [00:00<00:00,  5.53it/s]


{'eval_loss': 1.116915225982666,
 'eval_accuracy': 1.0,
 'eval_f1': 1.0,
 'eval_precision': 1.0,
 'eval_recall': 1.0,
 'eval_runtime': 1.1507,
 'eval_samples_per_second': 260.711,
 'eval_steps_per_second': 4.345,
 'epoch': 3.0}

In [5]:
trainer.save_model("../../model/intentClassifier")
tokenizer.save_pretrained("../../model/intentClassifier")

Saving model checkpoint to ../../model/intentClassifier
Configuration saved in ../../model/intentClassifier/config.json
Model weights saved in ../../model/intentClassifier/pytorch_model.bin
tokenizer config file saved in ../../model/intentClassifier/tokenizer_config.json
Special tokens file saved in ../../model/intentClassifier/special_tokens_map.json


('../../model/intentClassifier/tokenizer_config.json',
 '../../model/intentClassifier/special_tokens_map.json',
 '../../model/intentClassifier/vocab.txt',
 '../../model/intentClassifier/added_tokens.json')

In [6]:
# Inference example
def predict_intent(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = model(**inputs)
        predicted_class_id = outputs.logits.argmax().item()
    return label_encoder.inverse_transform([predicted_class_id])[0]

In [8]:
# Example usage
print(predict_intent("How do I apply for scholarships?"))
print(predict_intent("I have a complaint about the cafeteria service?"))
print(predict_intent("Hi, I'm trying to figure out how to pay my tuition fees."))
print(predict_intent("Are there any upcoming student events"))

Scholarship/Financial Aid
General Complaint
Tuition/Fees Inquiry
Extracurricular Activities
